In [1]:
def get_vocab_size(corpus_file):
    with open(corpus_file, "r", encoding="utf-8") as f:
        tokens = set(" ".join(f.readlines()).split())
    return len(tokens)

In [2]:
from collections import Counter, defaultdict
import pandas as pd

def load_ngram_counts(file_path):
    counts = Counter()
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                ngram = tuple(parts[0].split())
                count = int(parts[1])
                counts[ngram] = count
    return counts

def generate_good_turing_table(counts, output_file):
    Nc_raw = defaultdict(int)
    for count in counts.values():
        Nc_raw[count] += 1

    N_total = sum(counts.values())  # total seen n-grams

    table = []
    for c in sorted(Nc_raw.keys())[:100]:
        Nc = Nc_raw[c]
        Nc_plus1 = Nc_raw.get(c + 1, 0)
        c_star = ((c + 1) * Nc_plus1 / Nc) if Nc > 0 and Nc_plus1 > 0 else c
        Nc_prob = Nc / N_total  # Good-Turing interpretation: probability mass for count c
        table.append((c, round(Nc_prob, 8), round(c_star, 4)))

    df = pd.DataFrame(table, columns=["C (MLE)", "Nc (GT Prob)", "C*"])
    df.to_csv(output_file, sep="\t", index=False)
    print(f"Saved Good-Turing frequency table to {output_file}")

In [3]:
# Compute vocabulary size from full corpus
V = get_vocab_size("tokenized_telugu.txt")

# Model files
models = {
    "unigram": "unigram_model.txt",
    "bigram": "bigram_model.txt",
    "trigram": "trigram_model.txt",
    "quadrigram": "quadrigram_model.txt"
}

# Generate frequency tables
for name, path in models.items():
    counts = load_ngram_counts(path)
    generate_good_turing_table(counts, f"{name}_frequency_table.txt")

Saved Good-Turing frequency table to unigram_frequency_table.txt
Saved Good-Turing frequency table to bigram_frequency_table.txt
Saved Good-Turing frequency table to trigram_frequency_table.txt
Saved Good-Turing frequency table to quadrigram_frequency_table.txt
